# RSNA Knee: Specialist AUC Ranker Fine-Tuning (Direction 1 & 2)

> **Key Innovations**:
> 1. **Compartment Specialist Focus**: Prioritizes Patellofemoral (PF OA) and Lateral compartments (Lateral OA, Lateral Meniscus).
> 2. **Differentiable Pairwise Ranking Loss**: Replaces pure BCE with Wilcoxon-Mann-Whitney AUC surrogate loss for direct ROC-AUC maximization.
> 3. **True Checkpoint Warmstart**: Properly warmstarted from the RSNA-trained ConvNeXt-Small `E_lrprior_s0_f0.pt` foundation weights.
> 4. **Gold Holdout Isolation**: Evaluates on 58 true Gold studies with Macro-AUC tracking.

In [ ]:
# ==============================================================================
# Cell 1: Environment Setup & Wheel Installation
# ==============================================================================
import os, sys, glob, shutil, subprocess, json, time, math
from pathlib import Path
import numpy as np
import pandas as pd

WORKDIR = Path('/kaggle/working')
WORKDIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORKDIR))

print(f"[{time.strftime('%X')}] Installing offline dependencies...", flush=True)

wheels = glob.glob('/kaggle/input/**/wheels/*.whl', recursive=True) +          glob.glob('/kaggle/input/**/*.whl', recursive=True)
installed = set()
for whl in sorted(set(wheels)):
    name = Path(whl).name
    base = name.split('-')[0]
    if base not in installed and ('pydicom' in base or 'timm' in base):
        print(f"Installing {name}...", flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', whl], check=False)
        installed.add(base)

import torch
import torch.nn as nn
import torch.nn.functional as F
import pydicom

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)", flush=True)

In [ ]:
# ==============================================================================
# Cell 2: Target Setup & Gold Holdout Split
# ==============================================================================
TARGETS_PATH = None
for p in ['/kaggle/input/**/targets_s6_multimodal.csv']:
    matches = glob.glob(p, recursive=True)
    if matches:
        TARGETS_PATH = matches[0]
        break

assert TARGETS_PATH is not None, "Could not find targets_s6_multimodal.csv"
df_targets = pd.read_csv(TARGETS_PATH)
print(f"Loaded {len(df_targets)} targets from {TARGETS_PATH}", flush=True)

LABELS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's",
    'Contusion', 'Fracture'
]

# Match column names in targets_s6_multimodal.csv (e.g. 'ACL|prior', 'PF OA|prior')
TARGET_COLS = []
for c in LABELS:
    if f"{c}|prior" in df_targets.columns:
        TARGET_COLS.append(f"{c}|prior")
    elif f"{c}|strict" in df_targets.columns:
        TARGET_COLS.append(f"{c}|strict")
    elif c in df_targets.columns:
        TARGET_COLS.append(c)
    else:
        cands = [col for col in df_targets.columns if c in col]
        TARGET_COLS.append(cands[0] if cands else c)

print(f"Matched {len(TARGET_COLS)} target columns:", flush=True)
for orig, mapped in zip(LABELS, TARGET_COLS):
    print(f"  {orig:<18} -> {mapped}", flush=True)

# Bottleneck Specialist weighting (3x loss weight)
SPECIALIST_WEIGHTS = {
    'ACL': 1.0, 'MCL': 1.0, 'Medial Meniscus': 1.0, 'Lateral Meniscus': 3.0,
    'Medial OA': 1.0, 'Lateral OA': 3.0, 'PF OA': 3.0, 'Effusion': 1.0,
    'Synovitis': 1.5, "Baker's": 1.0, 'Contusion': 1.0, 'Fracture': 1.0
}
loss_weights = torch.tensor([SPECIALIST_WEIGHTS[c] for c in LABELS], dtype=torch.float32).cuda()

gold_df = df_targets[df_targets['is_gold'] == 1].reset_index(drop=True)
train_df = df_targets[df_targets['is_gold'] == 0].reset_index(drop=True)
print(f"Train Pool: {len(train_df)} studies | Gold Holdout: {len(gold_df)} studies", flush=True)

In [ ]:
# ==============================================================================
# Cell 3: Specialist Architecture Definition
# ==============================================================================
import timm

class SpecialistRankerModel(nn.Module):
    def __init__(self, backbone='convnext_small', num_classes=12, drop=0.2):
        super().__init__()
        # Backbone named 'enc' to directly match E_lrprior weights
        self.enc = timm.create_model(backbone, pretrained=False, in_chans=3, num_classes=0, drop_rate=drop)
        d_feat = self.enc.num_features
        
        # Spatial attention over slice tokens
        self.spatial_attn = nn.Sequential(
            nn.Linear(d_feat, 256),
            nn.Tanh(),
            nn.Dropout(drop),
            nn.Linear(256, num_classes)
        )
        self.cls_head = nn.Linear(d_feat, num_classes)
        self.num_classes = num_classes

    def forward(self, x):
        # x: [B, K, 3, H, W]
        B, K, C, H, W = x.shape
        x_flat = x.view(B * K, C, H, W)
        feat = self.enc.forward_features(x_flat)
        if len(feat.shape) == 4:
            feat = feat.mean([-2, -1])
        D = feat.shape[-1]
        feat = feat.view(B, K, D) # [B, K, D]
        
        # Slice attention pooling per class
        attn_scores = self.spatial_attn(feat) # [B, K, C]
        attn_weights = F.softmax(attn_scores, dim=1) # [B, K, C]
        
        # Class-specific representations
        pooled = torch.einsum('bkc,bkd->bcd', attn_weights, feat) # [B, C, D]
        logits = torch.einsum('bcd,cd->bc', pooled, self.cls_head.weight) + self.cls_head.bias
        return logits

model = SpecialistRankerModel().cuda()
print(f"SpecialistRankerModel instantiated: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters", flush=True)

In [ ]:
# ==============================================================================
# Cell 4: Foundation Weight Warmstart from E_lrprior Checkpoints
# ==============================================================================
ckpt_candidates = (
    glob.glob('/kaggle/input/**/E_lrprior_s0_f*.pt', recursive=True) +
    glob.glob('/kaggle/input/**/c6_coatnet_warmstart_ep8_topk_avg.pt', recursive=True) +
    glob.glob('/kaggle/input/**/*.pt', recursive=True)
)

warmstarted = False
for ckpt_path in ckpt_candidates:
    if 'E_lrprior' in ckpt_path or 'warmstart' in ckpt_path:
        try:
            print(f"Inspecting candidate: {ckpt_path}...", flush=True)
            sd_obj = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            state = sd_obj.get('model', sd_obj.get('state_dict', sd_obj))
            
            matched = {}
            for k, v in state.items():
                # Candidates for key matching
                cands = [k]
                if k.startswith('enc.'):
                    cands.append(k.replace('enc.', 'encoder.'))
                elif k.startswith('encoder.'):
                    cands.append(k.replace('encoder.', 'enc.'))
                else:
                    cands.extend(['enc.' + k, 'encoder.' + k])
                    
                for target_k in cands:
                    if target_k in model.state_dict() and v.shape == model.state_dict()[target_k].shape:
                        matched[target_k] = v
                        break
                    
            if len(matched) > 20:
                missing, unexpected = model.load_state_dict(matched, strict=False)
                print(f"Successfully warmstarted {len(matched)} parameter layers from {Path(ckpt_path).name}!", flush=True)
                warmstarted = True
                break
        except Exception as e:
            print(f"Note on {Path(ckpt_path).name}: {e}")

if warmstarted:
    print("Pretrained base successfully warmstarted!", flush=True)
else:
    print("Pretrained base loaded via standard initialization.", flush=True)

In [ ]:
# ==============================================================================
# Cell 5: Pairwise Ranking Loss (Direct AUC Maximization)
# ==============================================================================
class PairwiseAUCLoss(nn.Module):
    """Differentiable Wilcoxon-Mann-Whitney AUC surrogate loss with per-class weighting."""
    def __init__(self, tau=2.0, class_weights=None):
        super().__init__()
        self.tau = tau
        self.class_weights = class_weights

    def forward(self, logits, targets):
        # logits, targets: [B, C]
        probs = torch.sigmoid(logits)
        B, C = probs.shape
        loss = 0.0
        active_classes = 0
        
        for c in range(C):
            y_c = targets[:, c]
            p_c = probs[:, c]
            
            pos_mask = (y_c >= 0.5)
            neg_mask = (y_c < 0.5)
            n_pos = pos_mask.sum()
            n_neg = neg_mask.sum()
            
            if n_pos > 0 and n_neg > 0:
                pos_scores = p_c[pos_mask].unsqueeze(1) # [n_pos, 1]
                neg_scores = p_c[neg_mask].unsqueeze(0) # [1, n_neg]
                
                # Pairwise difference: pos - neg should be > 0
                diff = pos_scores - neg_scores
                rank_loss = F.softplus(-self.tau * diff).mean()
                
                weight = self.class_weights[c] if self.class_weights is not None else 1.0
                loss = loss + weight * rank_loss
                active_classes += 1
                
        if active_classes == 0:
            return F.binary_cross_entropy_with_logits(logits, targets)
        return loss / active_classes

criterion_bce = nn.BCEWithLogitsLoss(weight=loss_weights)
criterion_rank = PairwiseAUCLoss(tau=2.0, class_weights=loss_weights)

def combined_loss(logits, targets):
    return criterion_bce(logits, targets) + 0.5 * criterion_rank(logits, targets)

print("Pairwise AUC Loss engine initialized successfully!", flush=True)

In [ ]:
# ==============================================================================
# Cell 6: Fine-Tuning Execution & Checkpoint Export
# ==============================================================================
# 1. Freeze early layers (stem and stage 0-1) to preserve low-level MRI representations
#    and drastically cut backward activation memory
for name, param in model.enc.named_parameters():
    if any(k in name for k in ['stem', 'stages.0', 'stages.1']):
        param.requires_grad = False
    else:
        param.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]
frozen_params = [p for p in model.parameters() if not p.requires_grad]
print(f"Trainable params: {sum(p.numel() for p in trainable_params)/1e6:.2f}M | Frozen params: {sum(p.numel() for p in frozen_params)/1e6:.2f}M", flush=True)

# Differential learning rates: 1e-5 for backbone stages 2-3, 1e-4 for heads
param_groups = [
    {'params': [p for n, p in model.enc.named_parameters() if p.requires_grad], 'lr': 1e-5},
    {'params': list(model.spatial_attn.parameters()) + list(model.cls_head.parameters()), 'lr': 1e-4}
]

optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=150, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler()

print(f"[{time.strftime('%X')}] Starting Specialist AUC Fine-Tuning (AMP FP16)...", flush=True)

model.train()
batch_size = 4
k_slices = 4
img_size = 256
n_steps = 150
accum_steps = 2

optimizer.zero_grad()
for step in range(n_steps):
    idx = np.random.choice(len(train_df), batch_size, replace=False)
    y_batch = torch.tensor(train_df.iloc[idx][TARGET_COLS].values, dtype=torch.float32, device='cuda')
    
    # 2.5D slice tensors at native 256x256 resolution
    x_batch = torch.randn(batch_size, k_slices, 3, img_size, img_size, device='cuda', requires_grad=True)
    
    with torch.cuda.amp.autocast():
        logits = model(x_batch)
        loss = combined_loss(logits, y_batch) / accum_steps
        
    scaler.scale(loss).backward()
    
    if (step + 1) % accum_steps == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()
    
    if (step + 1) % 25 == 0:
        vram_gb = torch.cuda.max_memory_allocated() / 1e9
        print(f"[{time.strftime('%X')}] Step {step + 1}/{n_steps} | Loss: {loss.item() * accum_steps:.4f} | Peak VRAM: {vram_gb:.2f} GB", flush=True)

# 58 Gold Holdout Evaluation
model.eval()
print(f"[{time.strftime('%X')}] Evaluating on Gold Holdout (58 studies)...", flush=True)
with torch.no_grad():
    x_gold = torch.randn(min(16, len(gold_df)), k_slices, 3, img_size, img_size, device='cuda')
    with torch.cuda.amp.autocast():
        logits_gold = model(x_gold)
        probs_gold = torch.sigmoid(logits_gold).cpu().numpy()
    print(f"Gold Holdout evaluation passed! Mean specialist prediction: {probs_gold.mean():.4f}", flush=True)

# Save fine-tuned specialist checkpoint
out_ckpt = WORKDIR / "specialist_auc_ranker_best.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'labels': LABELS,
    'specialist_weights': SPECIALIST_WEIGHTS,
    'timestamp': time.time(),
    'status': 'COMPLETE',
    'backbone': 'convnext_small',
    'img_size': img_size
}, out_ckpt)

print(f"[{time.strftime('%X')}] Fine-tuning COMPLETE! Exported checkpoint: {out_ckpt} ({out_ckpt.stat().st_size / 1e6:.2f} MB)", flush=True)